# 05 — Portfolio simulation, reconciliation and analytics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/a-quant-journey/blob/main/notebooks/teaching/05_vectorbt_portfolio_reconciliation.ipynb)

Can a second execution engine reproduce every cash movement in Notebook 04? We replay Notebook 03’s unchanged, dated validation predictions. Each engine starts with USD 1,000. This previously examined validation period supports an accounting check. It supplies no fresh estimate of predictive performance.

For local execution, use the nb05-native Python 3.11 kernel installed from `requirements-vectorbt.lock`. Set `AQJ_ARTIFACT_DIR` to the absolute `outputs/track_e_multifamily` directory inside the explicit run printed by the 03/04 teaching runner. Start the notebook within the public checkout.

**Google Colab:** Use the 2026.07 CPU runtime. On first execution, dependency installation may require a session restart. If prompted, restart the session and rerun the setup cell before continuing.

In [ ]:
# Colab only. Select a Python 3.12 CPU runtime.
import os

if (type(get_ipython()).__module__.startswith('google.colab.')
        and not os.environ.get('AQJ_COLAB_PREREQUISITE')):
    import runpy
    import subprocess
    from pathlib import Path

    branch = 'main'
    checkout = Path('/content/aqj-teaching') / branch
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--branch', branch, '--single-branch',
                        'https://github.com/FranQuant/a-quant-journey.git', str(checkout)], check=True)
    subprocess.run(['git', 'diff', '--quiet', 'HEAD'], cwd=checkout, check=True)
    runpy.run_path(str(checkout / 'scripts/colab_teaching.py'))['initialize'](5)

## 1. Verify the prediction handoff

The loader checks the prediction bytes, labels and feature dates against the bounded validation sample. The run identifier below names the selected teaching run. The bounded data cutoff is **2021-12-28**. The 500 signal closes are followed by two more observed closes so the final prediction has both an execution opportunity and a subsequent holding-period valuation.

In [ ]:
import hashlib
import platform
import os
import sys
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
# QuantStats sets a global plotting theme when VectorBT discovers it.
with plt.rc_context():
    import vectorbt

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/quant_journey').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start within the teaching checkout or runner snapshot.')
sys.path.insert(0, str(ROOT))
from src.quant_journey.backtesting.vectorbt_adapter import BENCHMARK, load_handoff, reconcile

artifacts = Path(os.environ['AQJ_ARTIFACT_DIR']).resolve(strict=True)
prices, signals = load_handoff(artifacts, ROOT / 'data/raw/tpq_eod_2014_2023.csv')
print('Python', platform.python_version(), '| VectorBT', vectorbt.__version__)
print('Prediction SHA-256', hashlib.sha256((artifacts / 'validation_predictions.csv').read_bytes()).hexdigest())
for label, dates in [('Signal', signals.index), ('Valuation', prices.index)]:
    print(f'{label}: {len(dates)} dates, {dates[0]:%Y-%m-%d} through {dates[-1]:%Y-%m-%d}')

In [ ]:
def diagram(nodes, links, size):
    fig, ax = plt.subplots(figsize=size)
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.axis('off')
    boxes = []
    for x, y, label, color in nodes:
        text = ax.text(x, y, label, ha='center', va='center', color='#17304a',
                       bbox=dict(boxstyle='round,pad=.65', fc=color, ec='#64798b'))
        boxes.append(text.get_bbox_patch())
    fig.canvas.draw()
    for start, end in links:
        ax.annotate('', nodes[end][:2], nodes[start][:2],
                    arrowprops=dict(arrowstyle='-|>', color='#516775',
                                    patchA=boxes[start], patchB=boxes[end], lw=1.4))
    plt.show()

diagram([
    (.15, .5, 'Notebook 03\nDated predictions', '#e7eef6'),
    (.50, .77, 'Canonical ledger\nMap signals · update balances', '#f9eddd'),
    (.50, .23, 'VectorBT\nMap targets · simulate orders', '#e0f1ef'),
    (.84, .5, 'Reconcile\nDated states and fills', '#e7eef6'),
], [(0, 1), (0, 2), (1, 3), (2, 3)], (10, 2.3))

**Figure 1. Independent accounting paths.** Both engines receive the same predictions and observed closes. Each computes its own executions and balances before comparison.

## 2. From a signal to a VectorBT order

A prediction observed after close $t$ sets the target at the **next observed SPY close**. Shares bought there first earn the move to the following close. The last close values any remaining shares.

The adapter maps target changes to orders independently. VectorBT sizes each fill at the execution close.

| Change | Percent order | Instruction |
|---|---:|---|
| Cash to long | +1 | Spend available cash including fees |
| Long to cash | −1 | Sell all shares |
| Unchanged | NaN | Submit no order |

```python
delta = state.target_position.diff().fillna(state.target_position)
orders = delta.where(delta != 0)
pf = vbt.Portfolio.from_orders(
    close=prices, price=prices, size=orders, size_type="percent", direction="longonly",
    init_cash=1000., fees=bps / 10000, fixed_fees=0., slippage=0., cash_sharing=False,
    min_size=0., max_size=np.inf, size_granularity=np.nan, reject_prob=0.,
    allow_partial=False, raise_reject=True, log=True)
fills = pf.orders.records
```

With available cash $C$, execution close $P$, shares sold $q$ and proportional fee $f$, a purchase spends the available cash inclusive of fees:

$$
q_{\mathrm{buy}}=\frac{C}{P(1+f)}.
$$

A sale credits net proceeds:

$$
C_{\mathrm{sale}}=qP(1-f).
$$

In [ ]:
diagram([
    (.16, .5, 'Close t\nObserve and predict', '#e7eef6'),
    (.50, .5, 'Close t+1\nExecute a target change', '#f9eddd'),
    (.84, .5, 'Close t+2\nValue the holding period', '#e0f1ef'),
], [(0, 1), (1, 2)], (10, 1.3))

**Figure 2. Execution timing.** The first price move earned by newly purchased shares runs from the execution close to the following observed close.

In [ ]:
states, summary, discrepancies = reconcile(artifacts, prices, signals, signals.columns)

### Follow four GaussianNB predictions

Read each row from the feature date to its execution opportunity. Each row has an observed SPY trading-session close. NaN in the order column instructs the engine to skip trading at that close. A NaN fill means the account already held the desired position. The signed order below is the change in the reconciled target, with zero changes shown as NaN.

In [ ]:
audit = states['GaussianNB', 'illustrative_5bps']
change = audit.target_position.diff().fillna(audit.target_position)
example = pd.DataFrame({
    'Feature date': audit.signal_date,
    'Prediction': audit.signal_date.map(signals['GaussianNB']),
    'Target (0/1)': audit.target_position,
    'Signed percent order': change.where(change.ne(0)),
    'Execution close ($)': prices,
    'Signed fill (shares)': audit.signed_shares_traded.where(audit.execution_count.gt(0)),
}).dropna(subset=['Feature date']).head(4)
example['Prediction'] = example['Prediction'].astype(int)
display(example.rename_axis('Execution date').round(6))

## 3. Follow the cash

Let $C_t$ and $q_t$ be closing cash and shares at observed close $P_t$. Closing portfolio equity is

$$
V_t=C_t+q_tP_t.
$$

At the same execution price, the fee $F_t$ accounts for the change in equity:

$$
V_t^{\mathrm{post}}=V_t^{\mathrm{pre}}-F_t.
$$

In [ ]:
display(audit.loc[audit.execution_count > 0,
                  ['signal_date','target_position','fill_price','signed_shares_traded',
                   'trade_notional','transaction_cost','cash','shares','pre_trade_equity','equity']].head(2))

### Compare all portfolios

Net return measures the change from initial capital. Maximum drawdown is the largest peak-to-trough equity decline, displayed as a positive percentage. Executed asset turnover sums absolute filled notional divided by pre-trade equity on each date. Its unit is multiples of equity over the full period. Each side of a round trip contributes once. Five basis points means 0.05% of notional per fill.

In [ ]:
scenarios = {'zero_cost': 'Zero fees', 'illustrative_5bps': '5 bps per fill'}
table = pd.DataFrame({
    'Portfolio': summary.portfolio, 'Fees per fill': summary.scenario.map(scenarios),
    'Initial ($)': 1000., 'Ending ($)': summary.ending_equity,
    'Net return (%)': 100 * summary['return'], 'Maximum drawdown magnitude (%)': -100 * summary.max_drawdown,
    'Fills': summary.trades, 'Fees ($)': summary.fees, 'Turnover (×)': summary.turnover,
})
for column in ['Initial ($)', 'Ending ($)', 'Fees ($)']:
    table[column] = table[column].map('${:,.2f}'.format)
display(table.set_index(['Portfolio', 'Fees per fill']).round(2))

In [ ]:
portfolios = [*signals.columns.drop(BENCHMARK), BENCHMARK]
returns_pct = summary.pivot(index='portfolio', columns='scenario', values='return').loc[portfolios] * 100
ax = returns_pct[list(scenarios)].rename(columns=scenarios).rename(
    index={BENCHMARK: 'Matched SPY (buy-and-hold)'}
).plot.barh(figsize=(9, 5), color=['#38759a', '#bc692d'], width=.78)
ax.invert_yaxis()
ax.axvline(0, color='black', lw=.9)
ax.set(xlabel='Cumulative net return (%)', ylabel='')
ax.legend(title='', loc='lower center', bbox_to_anchor=(.5, 1), ncol=2, frameon=False)
plt.tight_layout()
plt.show()

**Figure 3. Terminal returns.** Each pair follows one portfolio across the two fee assumptions. The vertical line marks the return that leaves starting capital intact.

In [ ]:
turnover = summary.loc[summary.scenario.eq('zero_cost')].set_index('portfolio').turnover
drag = returns_pct.zero_cost - returns_pct.illustrative_5bps
points = pd.DataFrame({'turnover': turnover, 'drag': drag}).loc[portfolios]
fig, ax = plt.subplots(figsize=(9, 4))
for (x, y), group in points.groupby(['turnover', 'drag'], sort=False):
    label = 'Logistic regression / majority / matched SPY' if len(group) == 3 else ', '.join(group.index)
    if len(group) == 3:
        assert set(group.index) == {'Logistic regression', 'Training majority baseline', BENCHMARK}
    ax.scatter(x, y, color='#386e89', s=36)
    ax.annotate(label, (x, y), xytext=(-6 if x > 190 else 6, 5), textcoords='offset points',
                ha='right' if x > 190 else 'left', fontsize=9)
ax.margins(x=.06, y=.15)
ax.set(xlabel='Cumulative executed asset turnover at zero fees (× equity)',
       ylabel='Terminal return drag (percentage points)')
fig.tight_layout()
plt.show()

**Figure 4. Turnover and cost drag.** The vertical difference is the zero-fee return minus the five-basis-point return, in percentage points. Fees also change the capital available for subsequent trades, so terminal return drag differs from the sum of fees paid in dollars. Each point compares the same dated strategy across scenarios.

## 4. Compare equity paths

The two invested paths shown are GaussianNB and matched SPY. Matched SPY starts in cash, buys at the first executable close and holds through the final valuation. The gray reference keeps the initial capital in cash at zero interest. On each strategy path, the dashed ledger line overlays the wider VectorBT line. When the engines agree, both trace the same path.

In [ ]:
saved = pd.read_csv(artifacts / 'portfolio_equity_curves.csv', header=[0, 1], index_col=0, parse_dates=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (scenario, title) in zip(axes, scenarios.items()):
    for name, color in [('GaussianNB', 'tab:blue'), (BENCHMARK, 'tab:orange')]:
        key = (name, scenario)
        ax.plot(prices.index, states[key].equity, color=color, lw=3, label=name)
        ax.plot(saved.index, saved[key], '--', lw=1, color='black')
    ax.axhline(1000, color='gray', ls=':', label='Constant cash')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=35)
axes[0].set_ylabel('Portfolio equity (USD)')
axes[0].plot([], [], '--', color='black', lw=1, label='Notebook 04 ledger overlay')
fig.legend(*axes[0].get_legend_handles_labels(), loc='upper center', ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0, 1, .90))
plt.show()

**Figure 5. Selected equity paths.** The black ledger dashes follow the colored VectorBT curves on a common USD scale. The dotted cash reference stays at its starting value.

## 5. Measure reconciliation error

For engine value $x_i$ and ledger reference $y_i$, each numeric comparison must satisfy

$$
|x_i-y_i|\le a+r|y_i|.
$$

| Fields | Absolute tolerance $a$ | Relative tolerance $r$ |
|---|---:|---:|
| Dates, target positions and execution counts | Exact | Exact |
| Shares, signed fill quantity, turnover, returns and drawdown | $10^{-12}$ | $10^{-12}$ |
| Cash, equity, prices, fees and trade notional | $10^{-9}$ | $10^{-12}$ |

Across observations with comparable values, the reported maxima are

$$
D_{\mathrm{abs}}=\max_i |x_i-y_i|,\qquad
D_{\mathrm{rel}}=\max_{i:y_i\ne0}\frac{|x_i-y_i|}{|y_i|}.
$$

Zero references still undergo the acceptance check. The relative diagnostic is zero when no nonzero reference is available. The two maxima can occur on different dates.

In [ ]:
display(discrepancies.groupby(level='field').max().map('{:.2e}'.format))
print(len(states), 'portfolio/scenario cases passed')
print(sum(len(s) for s in states.values()), 'dated states reconciled')

### Equity differences in dollars

Subtract the verified saved ledger equity from VectorBT equity on each date. The shaded interval spans the smallest to largest signed difference across the nine portfolios in each scenario. Read the scientific-notation multiplier beside the axis. Each annotation gives the largest absolute equity difference across dates and portfolios. The discrepancies are consistent with floating-point precision on accounts starting at USD 1,000.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), sharey=True)
for ax, (scenario, title) in zip(axes, scenarios.items()):
    residuals = pd.DataFrame({
        name: states[name, scenario].equity - saved[name, scenario] for name in portfolios
    })
    expected = {name for name, case in states if case == scenario}
    assert set(residuals.columns) == expected and residuals.shape == (len(prices), 9)
    assert residuals.notna().all().all()
    peak = residuals.abs().max().max()
    ax.text(.03, .09, f'9 portfolios · max |error| = {peak:.2e} USD',
            transform=ax.transAxes, fontsize=9, bbox=dict(facecolor='white', edgecolor='none'))
    ax.fill_between(residuals.index, residuals.min(axis=1), residuals.max(axis=1), alpha=.45)
    ax.axhline(0, color='gray', lw=.7)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useOffset=False)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=35)
axes[0].set_ylabel('VectorBT − ledger (USD)')
fig.tight_layout()
plt.show()

**Figure 6. Signed equity residuals.** The band includes matched SPY and all eight prediction portfolios. Both panels use the same scale in USD.

## 6. Portfolio analytics with VectorBT and QuantStats

The ledger supplies an independent reference for cash, shares and equity. We now inspect the VectorBT object that produces those balances for the same GaussianNB illustration at five basis points per fill. QuantStats takes the reconciled daily net returns as its input.

Each observed close counts as one trading observation. We use 252 observations per year, sample volatility, and zero risk-free and required returns. The frequency arguments below set the annualization factor while preserving the observed timestamps and all cash days.

In [ ]:
import numpy as np
from src.quant_journey.execution.prediction_handoff import INITIAL_CASH
from src.quant_journey.backtesting.vectorbt_adapter import simulate_with_portfolio

state, pf = simulate_with_portfolio(prices, signals['GaussianNB'], 5)
pd.testing.assert_frame_equal(state, audit, check_exact=True)
returns = pf.returns()
matched = states[BENCHMARK, 'illustrative_5bps']
bm = matched.net_return
assert returns.index.equals(bm.index) and returns.index.equals(state.index)
assert len(returns) == 502 and np.isfinite(returns).all()
np.testing.assert_allclose(returns, state.net_return, atol=1e-12, rtol=1e-12)

In [ ]:
acc = pf.get_returns_acc(benchmark_rets=bm, freq='1D', year_freq='252D',
                         defaults=dict(risk_free=0., required_return=0., ddof=1))
assert acc.ann_factor == 252
dd = state.equity / state.equity.cummax() - 1
closed = pf.exit_trades.closed
stats = pf.stats(metrics=['start_value', 'end_value', 'total_return', 'total_fees_paid',
                          'max_dd', 'win_rate', 'profit_factor', 'expectancy',
                          'avg_winning_trade', 'avg_losing_trade'],
                 settings=dict(trades_type='exittrades', incl_open=False))

In [ ]:
display(pd.DataFrame([
    ('Initial equity', f"${stats['Start Value']:,.2f}", 'Fees paid', f"${stats['Total Fees Paid']:,.2f}"),
    ('Ending equity', f"${stats['End Value']:,.2f}", 'Filled orders', pf.orders.count()),
    ('Cumulative net return', f"{stats['Total Return [%]']:.2f}%", 'Closed / open trades',
     f"{closed.count()} / {pf.exit_trades.open.count()}"),
    ('Annualized return', f'{acc.annualized():.2%}', 'Closed win rate', f"{stats['Win Rate [%]']:.2f}%"),
    ('Annualized volatility', f'{acc.annualized_volatility():.2%}', 'Profit factor', f"{stats['Profit Factor']:.2f}"),
    ('Sharpe', f'{acc.sharpe_ratio():.2f}', 'Expectancy per closed trade', f"${stats['Expectancy']:.2f}"),
    ('Sortino', f'{acc.sortino_ratio():.2f}', 'Mean winning return', f"{stats['Avg Winning Trade [%]']:.2f}%"),
    ('Maximum drawdown magnitude', f"{stats['Max Drawdown [%]']:.2f}%", 'Mean losing return',
     f"{stats['Avg Losing Trade [%]']:.2f}%"),
    ('Calmar', f'{acc.calmar_ratio():.2f}', 'Mean closed duration', f'{closed.duration.mean():.1f} trading intervals'),
    ('Position coverage', f'{pf.position_coverage():.2%}', '', '')
], columns=['Portfolio', 'Value', 'Trading', 'Result']).set_index('Portfolio'))

A filled order changes the portfolio’s shares. Entry and exit trades attribute P&L to purchases and sales, respectively. A position groups a continuous holding episode. Here each completed round trip consists of one buy and one sell. Closed P&L includes both executed fees. The open position carries its entry fee and is valued at the terminal close.

In [ ]:
orders = pf.orders.records_readable
trades = pf.exit_trades.records_readable
open_positions = pf.positions.open.records_readable
assert (len(orders), orders.Side.eq('Buy').sum(), orders.Side.eq('Sell').sum(),
        closed.count(), len(open_positions)) == (67, 34, 33, 33, 1)
realized = closed.pnl.sum()
unrealized = pf.positions.open.pnl.sum()
np.testing.assert_allclose(realized + unrealized, state.equity.iloc[-1] - INITIAL_CASH,
                           atol=1e-9, rtol=1e-12)
display(orders[['Timestamp', 'Side', 'Size', 'Price', 'Fees']].head(3))
fields = ['Entry Timestamp', 'Exit Timestamp', 'Entry Fees', 'Exit Fees', 'PnL', 'Status']
display(pd.concat([closed.records_readable.head(1)[fields], open_positions[fields]],
                  keys=['Completed round trip', 'Terminal mark']).droplevel(1))
print(f'Closed P&L ${realized:,.2f}  |  Open P&L ${unrealized:,.2f}')

In [ ]:
import plotly.graph_objects as go

def plot_matched_returns(fig, add_trace_kwargs, xref, yref):
    # The returns accessor accepts our dated, fee-matched benchmark in VectorBT 0.28.5.
    acc.plot_cumulative(benchmark_rets=bm, start_value=0, fig=fig,
                        add_trace_kwargs=add_trace_kwargs, xref=xref, yref=yref,
                        main_kwargs=dict(trace_kwargs=dict(name='GaussianNB')),
                        benchmark_kwargs=dict(trace_kwargs=dict(name='Matched SPY')))

dashboard = go.Figure(pf.plot(
    subplots=['orders', 'trade_pnl', ('matched_returns', dict(
        title='Cumulative net return · 5 bps per fill', plot_func=plot_matched_returns))],
    subplot_settings=dict(trade_pnl=dict(
        pct_scale=False, closed_loss_trace_kwargs=dict(marker_symbol='x'),
        open_trace_kwargs=dict(marker_symbol='diamond-open'))),
    height=800, width=None, margin=dict(l=85, r=35, t=160, b=65)))
dashboard.update_yaxes(title_text='SPY close (USD)', row=1, col=1)
dashboard.update_yaxes(title_text='Trade P&L (USD)', row=2, col=1)
dashboard.update_yaxes(title_text='Net return', tickformat='.0%', row=3, col=1)
dashboard.update_layout(autosize=True, legend=dict(orientation='h', x=0, xanchor='left', y=1.10, yanchor='bottom'))
dashboard.update_xaxes(title_text='Observed SPY session date', row=3, col=1)
for annotation, title in zip(dashboard.layout.annotations,
                             ['SPY close and fills', 'Trade P&L', 'Cumulative net return · 5 bps per fill']):
    annotation.text = title

In [ ]:
import json
display({'application/vnd.plotly.v1+json': json.loads(dashboard.to_json()),
         'text/html': dashboard.to_html(full_html=False, include_plotlyjs=True,
                                        include_mathjax=False, config=dict(responsive=True))}, raw=True)

**Figure 7. Native VectorBT portfolio dashboard.** Buy and sell markers identify executable closes. Circles and crosses show profitable and losing closed trades; the open diamond shows the terminal mark. The bottom panel compares cumulative net returns from the same initial cash balance. Hover for values or zoom to inspect a holding period.

In [ ]:
import quantstats as qs

# QuantStats prepends a base one calendar day before its first input.
# For these dates it restores the genuine opening cash observation exactly.
assert returns.iloc[0] == 0 and returns.index[1] - returns.index[0] == pd.Timedelta(days=1)
snapshot_input = returns.iloc[1:].copy()
assert snapshot_input.index[0] - pd.Timedelta(days=1) == state.index[0]
snapshot = qs.plots.snapshot(snapshot_input, mode='comp', subtitle=False, show=False,
                             title='GaussianNB · net of 5 bps per fill', fontname='DejaVu Sans')
for axis, expected in zip(snapshot.axes, [state.equity / INITIAL_CASH - 1, dd, state.net_return], strict=True):
    line = axis.lines[0]
    assert pd.DatetimeIndex(line.get_xdata()).equals(state.index)
    np.testing.assert_allclose(line.get_ydata() / 100, expected, atol=1e-12, rtol=1e-12)
display(snapshot)
plt.close(snapshot)

**Figure 8. QuantStats performance snapshot.** The three panels follow cumulative net return, signed drawdown from the running equity peak, and returns between observed closes. The negative drawdown trough corresponds to the positive maximum-drawdown magnitude in the table.

In [ ]:
monthly = (1 + returns).groupby(returns.index.to_period('M')).prod() - 1
assert monthly.index.equals(pd.period_range('2020-01', '2021-12', freq='M'))
assert returns.index[-1] == pd.Timestamp('2021-12-28')
heatmap = qs.plots.monthly_heatmap(returns.copy(), compounded=True, eoy=False, active=False,
                                   returns_label='GaussianNB', figsize=(10, 3),
                                   fontname='DejaVu Sans', show=False)
matrix = heatmap.axes[0].collections[0].get_array().reshape(2, 12)
np.testing.assert_allclose(matrix / 100, monthly.to_numpy().reshape(2, 12),
                           atol=1e-12, rtol=1e-12)
display(heatmap)
plt.close(heatmap)

**Figure 9. QuantStats monthly return heatmap.** Each cell compounds the observed daily net returns in that month. December 2021 ends on December 28. Each printed percentage is readable independently of its color.

## Conclusion

Agreement across the two engines establishes accounting equivalence under the stated assumptions. It provides no evidence of alpha. Classification accuracy counts correct labels, while portfolio returns depend on the size and timing of price moves and the cost of switching. Logistic regression and the training-majority baseline predict UP throughout and match the always-long SPY benchmark in both scenarios.

After fees, no classifier beats matched SPY’s 47.83% cumulative return. GaussianNB moves from 49.04% before fees to 44.13% after fees.

Frequent switching incurs repeated fees and leaves less capital for later trades. Across 246 fills, the MLP ends at USD 1,026.52 with zero fees and USD 907.71 at five basis points per fill. The MLP experiences a smaller drawdown than matched SPY but finishes below its initial capital after fees. Drawdown measures the deepest decline from a peak and cannot establish superior risk-adjusted performance on its own. Constant cash is a visual reference outside the 18 reconciled cases.

These validation dates were previously examined. Hypothetical fills assume fractional liquidity at the observed close, zero spread and slippage, and no separate dividends or splits. Historical closes alone cannot establish real-world execution feasibility. The native records attribute the ending gain to completed round trips and the final open holding. The QuantStats panels describe how that same reconciled gain accumulated over time.